# Módulo 5 — Automatización de procesos

> Curso de Python para Análisis de Datos en Salud Pública — OPS/PAHO
> Material octubre 2022, actualizado 2026.
> Licencia: código MIT, contenido CC-BY-SA 4.0.

## Objetivos

Convertir un análisis exploratorio en un **pipeline reproducible** que corra sin intervención humana:

1. Pasar de notebook a script `.py`.
2. Estructurar el código en funciones.
3. Programar la ejecución con cron, Task Scheduler o GitHub Actions.

El ejemplo descarga datos COVID-19 globales de la OMS y produce un resumen Top-N por país.

## 1. De notebook a script

Un script `.py` se diferencia del notebook en cuatro cosas que importan para automatización:

| Aspecto | Notebook | Script |
|---------|----------|--------|
| Ejecución | celdas en orden manual | un solo `python archivo.py` |
| Argumentos | constantes hardcoded | flags CLI (`argparse`) |
| Errores | se ven en celda | se loggean y devuelven exit code |
| Reusabilidad | copy-paste | `import` desde otros scripts |

El script de ejemplo vive en [`src/paho_course/automation_example.py`](../src/paho_course/automation_example.py). Inspeccionémoslo:

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from paho_course import automation_example
print('Pipeline cargado desde:', automation_example.__file__)

## 2. Ejecutar el pipeline desde el notebook

Aunque el objetivo es correrlo automáticamente, conviene primero validarlo manualmente:

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

exit_code = automation_example.run(top_n=10)
print('Exit code:', exit_code)

In [ ]:
import pandas as pd
summary = pd.read_csv(REPO_ROOT / 'data' / 'processed' / 'covid_summary.csv')
summary

## 3. Ejecutar desde la línea de comandos

```bash
# Una corrida estándar
python -m paho_course.automation_example

# Cambiar el Top-N
python -m paho_course.automation_example --top 25

# Modo verbose
python -m paho_course.automation_example -v
```

El exit code (`0` éxito, `1` fallo) es lo que cron/Task Scheduler usan para detectar errores.

## 4. Programar la ejecución

### Opción A — cron (Linux / macOS)

Editar la tabla de cron con `crontab -e` y agregar:

```cron
# Correr todos los días a las 06:00
0 6 * * * cd /ruta/al/repo && /ruta/al/repo/.venv/bin/python -m paho_course.automation_example >> /var/log/paho_course.log 2>&1
```

Claves:
- usar **rutas absolutas** (cron no conoce el `PATH` del usuario)
- redirigir stdout/stderr a un log
- activar el venv con la ruta directa al `python` del venv

### Opción B — Task Scheduler (Windows)

1. Abrir **Programador de tareas** → **Crear tarea básica**.
2. Desencadenador: **Diariamente** a la hora deseada.
3. Acción: **Iniciar un programa**.
   - Programa: `C:\ruta\al\repo\.venv\Scripts\python.exe`
   - Argumentos: `-m paho_course.automation_example`
   - Iniciar en: `C:\ruta\al\repo`
4. Marcar **Ejecutar con los privilegios más altos** si el destino requiere acceso restringido.

### Opción C — GitHub Actions (recomendado para datos públicos)

El workflow [`.github/workflows/automation.yml`](../.github/workflows/automation.yml) corre el pipeline en la infraestructura de GitHub, sin servidor propio. Útil cuando:

- los datos son públicos (no hay credenciales sensibles)
- el output puede commitearse o publicarse como artifact
- se quiere historial de corridas con logs y alertas por email

## 5. Buenas prácticas

- **Idempotencia:** la segunda corrida con los mismos inputs debe producir el mismo output. Esto evita estados corruptos cuando el job se reintenta.
- **Logging estructurado:** usar `logging` (no `print`) con timestamps y niveles. Los operadores filtran por `ERROR` al revisar.
- **Exit codes:** `0` éxito, `≠0` falla. Cron/Task Scheduler/CI usan este código para alertar.
- **No commitear secretos:** si el pipeline necesita API keys, usar variables de entorno y `python-dotenv`; nunca hardcoded.
- **Tests pequeños:** una función `summarize(df, top_n)` se testea con un DataFrame de 5 filas en un segundo. Notebooks no se testean fácil; scripts sí.
- **Versionar outputs como datos:** si el output es una métrica histórica, considerar guardar con timestamp en el nombre: `covid_summary_2026-05-24.csv`.

## 6. Ejercicios

1. Modificar `summarize()` para que reporte también la **tasa de mortalidad** (`cumulative_deaths / cumulative_cases`).
2. Agregar un flag `--region` que filtre por región de la OMS (AMRO, EURO, etc.).
3. Cambiar la fuente: usar el dataset de egresos hospitalarios del Ecuador para generar un Top-10 de causas de egreso por mes.
4. Configurar el workflow de GitHub Actions para que corra los lunes a las 07:00 UTC y commitee el CSV resultante en una rama `data-updates`.